# Step 2: Process Features

## Traitement des attributs

In [5]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading pedestrian segments...


In [ ]:

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()
print('Boucle sur chaque attribut... peut prendre du temps (15-20mn)')

# Boucle sur le derniers attributs
for _, row in attributs_info.iterrows():
    if row['include_in_index']:
        attribute_name = row['attribute']
        method = row['method']
        how = row['how']
        value_column = row['value_column']
        buffer_size = row['buffer_size']
        geometry_type = row['geometry_type']
        feature_query_expr = make_feature_query(
        row.get('filter_column'),
        row.get('filter_values')
    )

        # Charger la couche attribut depuis gpd_attributs
        attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

        # Appliquer la méthode
        if method == "A": # Buffer feature extraction
            attribute_df = extract_buffer_feature(
                segments_gdf = segmented_net,
                feature_gdf = attribute_gdf,
                feature_name=attribute_name,
                geom_kind=geometry_type,
                buffer_radius=buffer_size,
                how=how,
                value_column=value_column,
                crs_meter_epsg=operation_crs,
                feature_query  = feature_query_expr,
            )
            print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
                # Debug duplicates
            if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
                print("\nDEBUG: Found duplicate segment assignments")
                dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
                print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
                # Keep only the first occurrence for each segment_id
                attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
                print("Dropped duplicates, keeping first occurrence")
            print(f"Spatial join computed for {attribute_name} with method {method}") 
        
        # Ajoute d'autres méthodes si besoin


        # Ajouter la colonne au GeoDataFrame principal
        segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)

# Sauvegarder
segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


Boucle sur chaque attribut... peut prendre du temps (15-20mn)
Buffer feature extracted for arbre_isole with method A
Spatial join computed for arbre_isole with method A
Buffer feature extracted for espace_vert with method A
Spatial join computed for espace_vert with method A
Buffer feature extracted for accident with method A
Spatial join computed for accident with method A
Buffer feature extracted for zone_apaisee with method A
Spatial join computed for zone_apaisee with method A
Buffer feature extracted for zone_pietonne with method A
Spatial join computed for zone_pietonne with method A
Buffer feature extracted for vitesse with method A
Spatial join computed for vitesse with method A
Buffer feature extracted for eau with method A
Spatial join computed for eau with method A
Buffer feature extracted for rez_actif with method A
Spatial join computed for rez_actif with method A
Buffer feature extracted for stationnement_genant with method A
Spatial join computed for stationnement_genant

In [9]:

segmented_net.head(20)


,Vitesse,geometry,segment_id,length,arbre_isole_A_10,espace_vert_A_10,accident_A_10,zone_apaisee_A_10,zone_pietonne_A_10,vitesse_A_10,eau_A_10,rez_actif_A_10,stationnement_genant_A_10,bruit_A_10,tp_A_10,amenite_A_10,espaces_ouverts_A_10,temperature_A_10,connectivite_A_10,largeur_trottoir_A_10,topographie_A_10
0,50,"LINESTRING (2505952.43 1117556.983, 2505957.52...",000000,18.611510,0.0,0.006828,0.0,4.042428,0.000000,4.042428,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.302301,16,2.0,3.1
1,50,"LINESTRING (2504875.759 1116854.188, 2504891.4...",000001,17.373877,0.0,0.000000,1.0,5.646604,0.000000,5.646604,0.0,0.0,0.0,0.0,0.0,0.0,1.0,33.389618,252,4.0,5.6
2,50,"LINESTRING (2500026.558 1117819.302, 2500041.9...",000002,50.000000,0.0,0.000000,8.0,1.008712,0.060493,1.008712,0.0,2.0,6.0,0.0,0.0,2.0,4.0,32.955685,963,9.0,62.9
3,50,"LINESTRING (2500045.896 1117773.338, 2500047.9...",000003,7.198671,0.0,0.000000,5.0,0.301179,0.055699,0.301179,0.0,0.0,1.0,0.0,0.0,0.0,2.0,33.564632,637,7.0,21.9
4,0,"LINESTRING (2498574.627 1115881.289, 2498530.0...",000004,45.125485,0.0,0.007477,0.0,0.000000,0.501680,0.000000,0.0,1.0,0.0,0.0,0.0,1.0,2.0,32.724442,621,10.0,49.6
5,50,"LINESTRING (2503779.53 1116266.438, 2503790.87...",000005,11.749078,0.0,0.200161,1.0,1.117355,0.143946,1.117355,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.600540,98,3.0,12.2
6,30,"LINESTRING (2496932.543 1119478.184, 2496913.2...",000006,22.405256,0.0,0.000000,0.0,6.260936,0.000000,4.006999,0.0,0.0,0.0,0.0,0.0,0.0,1.0,33.840801,24,6.0,27.3
7,30,"LINESTRING (2504805.799 1117155.916, 2504813.8...",000007,12.111091,0.0,0.000000,1.0,1.629306,0.000000,0.016417,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.890825,7,0.0,10.2
8,80,"LINESTRING (2498974.661 1122552.258, 2498967.7...",000008,18.010532,0.0,0.304132,1.0,19.966988,0.000000,19.966988,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.516232,48,0.0,49.3
9,50,"LINESTRING (2499069.719 1114426.012, 2499064.2...",000009,14.050916,0.0,0.000000,2.0,1.250200,0.000000,1.250200,0.0,0.0,0.0,0.0,0.0,0.0,2.0,32.646667,156,7.0,36.7


In [10]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
